- F1 Live AI Commentator - Entrainement du Modele ML avec Hyperparameter Tuning

**Objectif :** Predire le Top 3 d'une course F1 en temps reel

**Approche :**
- Modele de regression pour predire `final_position`
- Hyperparameter tuning avec RandomizedSearchCV
- Conversion en probabilites Top 3 / victoire
- Comparaison XGBoost vs Random Forest vs LightGBM

## Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from scipy.stats import randint, uniform

# Sauvegarde modele
import joblib
import pickle
import time

# Config
plt.style.use('ggplot')
sns.set_palette("husl")
pd.set_option('display.max_columns', 50)


In [2]:
# Chemins
DATA_PATH = Path("../data/processed/training_dataset.csv")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## Chargement des donnees

In [3]:
df = pd.read_csv(DATA_PATH)

print(f"Dataset charge : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"\nPeriode : {df['year'].min()} - {df['year'].max()}")
print(f"Courses uniques : {df['raceId'].nunique():,}")
print(f"Pilotes uniques : {df['driverId'].nunique():,}")

df.head()

Dataset charge : 551,742 lignes x 31 colonnes

Periode : 1996 - 2023
Courses uniques : 510
Pilotes uniques : 139


,raceId,driverId,constructorId,circuitId,year,round,lap,total_laps,laps_remaining,race_progress_pct,current_position,grid_position,positions_gained,last_lap_time_ms,avg_lap_time_ms,best_lap_time_ms,lap_time_std,is_fastest_lap,nb_pit_stops,last_pit_lap,laps_since_pit,driver_season_points,driver_season_pos,driver_wins_season,constructor_points,constructor_pos,driver_avg_pos_circuit,driver_wins_circuit,constructor_avg_circuit,final_position,is_dnf
0,1,1,1,1,2009,1,1,58,57,1.72,13,15,2,109088.0,109088.000000,109088.0,0.000000,0,0,0,1,0.0,0.0,0.0,0.0,0.0,4.933333,2,8.576923,20,1
1,1,1,1,1,2009,1,2,58,56,3.45,12,15,3,93740.0,101414.000000,93740.0,10852.674878,0,0,0,2,0.0,0.0,0.0,0.0,0.0,4.933333,2,8.576923,20,1
2,1,1,1,1,2009,1,3,58,55,5.17,11,15,4,91600.0,98142.666667,91600.0,9539.137347,0,0,0,3,0.0,0.0,0.0,0.0,0.0,4.933333,2,8.576923,20,1
3,1,1,1,1,2009,1,4,58,54,6.90,10,15,5,91067.0,96373.750000,91067.0,8554.512976,0,0,0,4,0.0,0.0,0.0,0.0,0.0,4.933333,2,8.576923,20,1
4,1,1,1,1,2009,1,5,58,53,8.62,10,15,5,92129.0,95524.800000,91067.0,7647.767629,0,0,0,5,0.0,0.0,0.0,0.0,0.0,4.933333,2,8.576923,20,1


## Exploration rapide (EDA)

In [4]:
# Info generales
print("Statistiques target (final_position) :")
print(df['final_position'].describe())

print("\nValeurs manquantes :")
missing = df.isnull().sum()
print(missing[missing > 0] if (missing > 0).any() else "Aucune")

Statistiques target (final_position) :
count    551742.00000
mean          9.67400
std           5.55867
min           1.00000
25%           5.00000
50%           9.00000
75%          14.00000
max          24.00000
Name: final_position, dtype: float64

Valeurs manquantes :
Aucune


## Feature Engineering

In [5]:
# Features derivees
df['lap_time_delta'] = df['last_lap_time_ms'] - df['avg_lap_time_ms']
df['pace_vs_best'] = df['last_lap_time_ms'] - df['best_lap_time_ms']
df['consistency'] = df['lap_time_std'] / df['avg_lap_time_ms']
df['is_early_race'] = (df['race_progress_pct'] < 33).astype(int)
df['is_mid_race'] = ((df['race_progress_pct'] >= 33) & (df['race_progress_pct'] < 66)).astype(int)
df['is_late_race'] = (df['race_progress_pct'] >= 66).astype(int)
df['position_momentum'] = df['grid_position'] - df['current_position']
df['has_pitted'] = (df['nb_pit_stops'] > 0).astype(int)
df['pit_frequency'] = df['nb_pit_stops'] / (df['lap'] + 1)

# Target binaire pour Top 3
df['is_top3'] = (df['final_position'] <= 3).astype(int)
df['is_winner'] = (df['final_position'] == 1).astype(int)

print(f"Features creees. Nouvelles colonnes : {df.shape[1]}")
print(f"\nDistribution Top 3 :")
print(df['is_top3'].value_counts(normalize=True))

Features creees. Nouvelles colonnes : 42

Distribution Top 3 :
is_top3
0    0.830635
1    0.169365
Name: proportion, dtype: float64


## Preparation Train/Val/Test Split

In [6]:
# Features a utiliser
EXCLUDE_COLS = [
    'raceId', 'driverId', 'constructorId', 'circuitId',
    'final_position', 'is_dnf', 'is_top3', 'is_winner',
]

feature_cols = [col for col in df.columns if col not in EXCLUDE_COLS]

print(f"Features utilisees : {len(feature_cols)}")
print(feature_cols)

Features utilisees : 34
['year', 'round', 'lap', 'total_laps', 'laps_remaining', 'race_progress_pct', 'current_position', 'grid_position', 'positions_gained', 'last_lap_time_ms', 'avg_lap_time_ms', 'best_lap_time_ms', 'lap_time_std', 'is_fastest_lap', 'nb_pit_stops', 'last_pit_lap', 'laps_since_pit', 'driver_season_points', 'driver_season_pos', 'driver_wins_season', 'constructor_points', 'constructor_pos', 'driver_avg_pos_circuit', 'driver_wins_circuit', 'constructor_avg_circuit', 'lap_time_delta', 'pace_vs_best', 'consistency', 'is_early_race', 'is_mid_race', 'is_late_race', 'position_momentum', 'has_pitted', 'pit_frequency']


In [7]:
# Split temporel (1996-2023)
train_df = df[df['year'] <= 2019].copy()
val_df = df[(df['year'] >= 2020) & (df['year'] <= 2021)].copy()
test_df = df[df['year'] >= 2022].copy()

print(f"Train : {len(train_df):,} lignes (1996-2019)")
print(f"Val   : {len(val_df):,} lignes (2020-2021)")
print(f"Test  : {len(test_df):,} lignes (2022-2023)")

# Verification test set non vide
if len(test_df) == 0:
    print("\nTest set vide ! Ajustement...")
    train_df = df[df['year'] <= 2018].copy()
    val_df = df[(df['year'] >= 2019) & (df['year'] <= 2020)].copy()
    test_df = df[df['year'] >= 2021].copy()
    print(f"Train : {len(train_df):,} lignes (1996-2018)")
    print(f"Val   : {len(val_df):,} lignes (2019-2020)")
    print(f"Test  : {len(test_df):,} lignes (2021-2023)")

# Separation X/y
X_train = train_df[feature_cols]
y_train = train_df['final_position']

X_val = val_df[feature_cols]
y_val = val_df['final_position']

X_test = test_df[feature_cols]
y_test = test_df['final_position']

print(f"\nShapes : X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")

Train : 472,504 lignes (1996-2019)
Val   : 42,088 lignes (2020-2021)
Test  : 37,150 lignes (2022-2023)

Shapes : X_train=(472504, 34), X_val=(42088, 34), X_test=(37150, 34)


## Hyperparameter Tuning avec RandomizedSearchCV

RandomizedSearchCV est plus rapide que GridSearchCV pour explorer un large espace d'hyperparametres.

In [8]:
# ── XGBoost Hyperparameter Tuning ───────────────────────────────────────────
print("Demarrage tuning XGBoost...")

# Grille de recherche
xgb_param_dist = {
    'n_estimators': [300, 500, 700],
    'max_depth': [6, 8, 10, 12],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.01, 0.1],
    'reg_lambda': [1, 1.5, 2]
}

xgb_base = xgb.XGBRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# RandomizedSearchCV (plus rapide que GridSearchCV)
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

xgb_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=50,  # 50 combinaisons aleatoires
    scoring=mae_scorer,
    cv=3,  # 3-fold CV temporel
    verbose=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start_time = time.time()
xgb_search.fit(X_train, y_train)
elapsed_time = time.time() - start_time

print(f"\nTuning XGBoost termine en {elapsed_time/60:.1f} minutes")
print(f"\nMeilleurs parametres XGBoost :")
print(xgb_search.best_params_)
print(f"\nMeilleur score CV (MAE) : {-xgb_search.best_score_:.3f}")

# Meilleur modele
xgb_model = xgb_search.best_estimator_

Demarrage tuning XGBoost...
Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.03, max_depth=8, min_child_weight=3, n_estimators=300, reg_alpha=0, reg_lambda=1, subsample=0.9; total time= 1.8min
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.03, max_depth=8, min_child_weight=3, n_estimators=300, reg_alpha=0, reg_lambda=1, subsample=0.9; total time= 1.9min
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.03, max_depth=8, min_child_weight=3, n_estimators=300, reg_alpha=0, reg_lambda=1, subsample=0.9; total time= 1.9min
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.01, max_depth=8, min_child_weight=1, n_estimators=300, reg_alpha=0, reg_lambda=1, subsample=0.7; total time= 2.1min
[CV] END colsample_bytree=0.8, gamma=0.1, learning_rate=0.01, max_depth=8, min_child_weight=1, n_estimators=300, reg_alpha=0, reg_lambda=1, subsample=0.7; total time= 2.2min
[CV] END colsample_bytree=0.8, gamma=0.1

In [ ]:
# ── LightGBM Hyperparameter Tuning ──────────────────────────────────────────
print("Demarrage tuning LightGBM...")

lgb_param_dist = {
    'n_estimators': [300, 500, 700],
    'max_depth': [6, 8, 10, 12, -1],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'num_leaves': [20, 31, 40, 50],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_samples': [10, 20, 30],
    'reg_alpha': [0, 0.01, 0.1],
    'reg_lambda': [0, 0.01, 0.1]
}

lgb_base = lgb.LGBMRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

lgb_search = RandomizedSearchCV(
    lgb_base,
    param_distributions=lgb_param_dist,
    n_iter=50,
    scoring=mae_scorer,
    cv=3,
    verbose=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start_time = time.time()
lgb_search.fit(X_train, y_train)
elapsed_time = time.time() - start_time

print(f"\nTuning LightGBM termine en {elapsed_time/60:.1f} minutes")
print(f"\nMeilleurs parametres LightGBM :")
print(lgb_search.best_params_)
print(f"\nMeilleur score CV (MAE) : {-lgb_search.best_score_:.3f}")

lgb_model = lgb_search.best_estimator_

Demarrage tuning LightGBM...
Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END colsample_bytree=0.9, learning_rate=0.03, max_depth=8, min_child_samples=10, n_estimators=300, num_leaves=20, reg_alpha=0.1, reg_lambda=0, subsample=0.9; total time=122.2min


In [ ]:
# ── Random Forest Hyperparameter Tuning (optionnel, plus lent) ─────────────
print("Demarrage tuning Random Forest...")

rf_param_dist = {
    'n_estimators': [200, 300, 400],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [5, 10, 15],
    'min_samples_leaf': [2, 5, 10],
    'max_features': ['sqrt', 'log2', None]
}

rf_base = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_search = RandomizedSearchCV(
    rf_base,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring=mae_scorer,
    cv=3,
    verbose=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start_time = time.time()
rf_search.fit(X_train, y_train)
elapsed_time = time.time() - start_time

print(f"\nTuning Random Forest termine en {elapsed_time/60:.1f} minutes")
print(f"\nMeilleurs parametres Random Forest :")
print(rf_search.best_params_)
print(f"\nMeilleur score CV (MAE) : {-rf_search.best_score_:.3f}")

rf_model = rf_search.best_estimator_

## Evaluation des modeles optimises

In [ ]:
# Predictions avec modeles optimises
models = {
    'XGBoost': xgb_model,
    'LightGBM': lgb_model,
    'Random Forest': rf_model
}

results = []

for name, model in models.items():
    print(f"\nEvaluation {name}...")
    
    # Predictions
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)
    
    # Metriques validation
    mae_val = mean_absolute_error(y_val, y_pred_val)
    rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
    r2_val = r2_score(y_val, y_pred_val)
    
    # Metriques test
    mae_test = mean_absolute_error(y_test, y_pred_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
    r2_test = r2_score(y_test, y_pred_test)
    
    print(f"{name} - Validation :")
    print(f"   MAE  : {mae_val:.3f} positions")
    print(f"   RMSE : {rmse_val:.3f}")
    print(f"   R2   : {r2_val:.3f}")
    
    print(f"\n{name} - Test :")
    print(f"   MAE  : {mae_test:.3f} positions")
    print(f"   RMSE : {rmse_test:.3f}")
    print(f"   R2   : {r2_test:.3f}")
    
    results.append({
        'Modele': name,
        'MAE_val': mae_val,
        'RMSE_val': rmse_val,
        'R2_val': r2_val,
        'MAE_test': mae_test,
        'RMSE_test': rmse_test,
        'R2_test': r2_test
    })

# Comparaison
results_df = pd.DataFrame(results)
print("\nComparaison des modeles optimises :")
print(results_df.to_string(index=False))

# Meilleur modele
best_idx = results_df['MAE_test'].idxmin()
best_model_name = results_df.loc[best_idx, 'Modele']
print(f"\nMeilleur modele : {best_model_name} (MAE test = {results_df.loc[best_idx, 'MAE_test']:.3f})")

best_model = models[best_model_name]

## Feature Importance

In [ ]:
# Feature importance
if hasattr(best_model, 'feature_importances_'):
    importances = pd.DataFrame({
        'feature': feature_cols,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)

    print("Top 20 features les plus importantes :")
    print(importances.head(20))

    # Visualisation
    plt.figure(figsize=(12, 8))
    top_features = importances.head(20)
    plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Importance')
    plt.title(f'Top 20 Features - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## Prediction Top 3 & Probabilites

In [ ]:
# Fonction de conversion position -> probabilite Top 3
def position_to_top3_prob(predicted_position):
    prob = 1 / (1 + np.exp((predicted_position - 2) / 1.5))
    return prob

# Calcul des probabilites
y_pred_test = best_model.predict(X_test)
test_df_eval = test_df.copy()
test_df_eval['predicted_position'] = y_pred_test
test_df_eval['top3_probability'] = test_df_eval['predicted_position'].apply(position_to_top3_prob)
test_df_eval['win_probability'] = (1 / (1 + np.exp((test_df_eval['predicted_position'] - 1) / 0.8)))

# Exemple : derniere course du test set
last_race = test_df_eval['raceId'].max()
race_example = test_df_eval[test_df_eval['raceId'] == last_race].copy()
race_example = race_example.sort_values('top3_probability', ascending=False)

print(f"\nExemple : Course raceId={last_race}")
print("\nTop 10 predictions (par probabilite Top 3) :")
print(race_example[[
    'driverId', 'lap', 'current_position', 'predicted_position', 
    'top3_probability', 'win_probability', 'final_position'
]].head(10))

In [ ]:
# Evaluation precision Top 3
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

test_df_eval['predicted_top3'] = (test_df_eval['predicted_position'] <= 3.5).astype(int)
test_df_eval['actual_top3'] = test_df_eval['is_top3']

acc = accuracy_score(test_df_eval['actual_top3'], test_df_eval['predicted_top3'])
prec = precision_score(test_df_eval['actual_top3'], test_df_eval['predicted_top3'])
rec = recall_score(test_df_eval['actual_top3'], test_df_eval['predicted_top3'])
f1 = f1_score(test_df_eval['actual_top3'], test_df_eval['predicted_top3'])

print(f"\nMetriques prediction Top 3 (test set) :")
print(f"   Accuracy  : {acc:.3f}")
print(f"   Precision : {prec:.3f}")
print(f"   Recall    : {rec:.3f}")
print(f"   F1-Score  : {f1:.3f}")

## Sauvegarde du Modele

In [ ]:
# Sauvegarde du meilleur modele
model_path = MODEL_DIR / f"f1_position_predictor_{best_model_name.lower().replace(' ', '_')}_tuned.pkl"
joblib.dump(best_model, model_path)

# Sauvegarde de la liste des features
features_path = MODEL_DIR / "feature_names.pkl"
joblib.dump(feature_cols, features_path)

# Sauvegarde des hyperparametres optimaux
if best_model_name == 'XGBoost':
    best_params = xgb_search.best_params_
elif best_model_name == 'LightGBM':
    best_params = lgb_search.best_params_
else:
    best_params = rf_search.best_params_

# Sauvegarde des metadonnees
metadata = {
    'model_name': best_model_name,
    'best_params': best_params,
    'mae_test': results_df.loc[best_idx, 'MAE_test'],
    'rmse_test': results_df.loc[best_idx, 'RMSE_test'],
    'r2_test': results_df.loc[best_idx, 'R2_test'],
    'features': feature_cols,
    'train_years': f"{train_df['year'].min()}-{train_df['year'].max()}",
    'test_years': f"{test_df['year'].min()}-{test_df['year'].max()}",
    'n_train_samples': len(train_df),
    'n_test_samples': len(test_df),
    'tuning_method': 'RandomizedSearchCV',
    'cv_folds': 3
}

metadata_path = MODEL_DIR / "model_metadata_tuned.pkl"
joblib.dump(metadata, metadata_path)

print(f"\nModele sauvegarde : {model_path}")
print(f"Features sauvegardees : {features_path}")
print(f"Metadonnees sauvegardees : {metadata_path}")
print(f"\nHyperparametres optimaux :")
for key, value in best_params.items():
    print(f"   {key}: {value}")

## Test de Chargement

In [ ]:
# Test de rechargement
loaded_model = joblib.load(model_path)
loaded_features = joblib.load(features_path)
loaded_metadata = joblib.load(metadata_path)

# Test prediction
test_pred = loaded_model.predict(X_test[loaded_features][:5])

print("Modele recharge avec succes !")
print(f"\nMetadonnees :")
for key, value in loaded_metadata.items():
    if key not in ['features', 'best_params']:
        print(f"   {key}: {value}")

print(f"\nTest prediction (5 premiers echantillons) :")
print(f"   Predictions : {test_pred}")
print(f"   Reel        : {y_test[:5].values}")